# Notebook-2: Embeddings & Vector Store

## Purpose: Convert text to vectors and enable semantic search

## Date: 11 Dec, 2025

### Author: Ram Prashanth Rao G

In [1]:
# Import Libraries

from sentence_transformers import SentenceTransformer
import numpy as np
from pathlib import Path
import sys

print("\n Imports are successful!")
print("Loading embedding model...")

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model Loaded: all-MiniLM-l6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print("\n Ready to generate Embeddings!")




 Imports are successful!
Loading embedding model...
Model Loaded: all-MiniLM-l6-v2
Embedding dimension: 384

 Ready to generate Embeddings!


In [2]:
### Testing out the sentences

test_sentences = [
    "SAP has a policy on AI ethics",
    "The company created guidelines for AI fairness",
    "Pizza is my favorite food",
    "Remote work requires secure internet connection"
]

print("Original Sentences:")
for i, sentence in enumerate(test_sentences):
    print(f"{i + 1}.{sentence}")

print("\n Generate Embeddings...")
embeddings = model.encode(test_sentences)

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding shape: {embeddings[0].shape}")
print(f"Dimension: {len(embeddings[0])} numbers per sentence")

print("\n First embedding (first 10 numbers):")
print(embeddings[0][:10])
print(" ...(374 more numbers)")

Original Sentences:
1.SAP has a policy on AI ethics
2.The company created guidelines for AI fairness
3.Pizza is my favorite food
4.Remote work requires secure internet connection

 Generate Embeddings...
Generated 4 embeddings
Each embedding shape: (384,)
Dimension: 384 numbers per sentence

 First embedding (first 10 numbers):
[-0.05288297 -0.01760507 -0.03363908 -0.02520042  0.05001527 -0.00385388
 -0.00209888  0.01754766  0.03065421  0.09484662]
 ...(374 more numbers)


In [3]:
### Calculate similarity between embeddings

from sklearn.metrics.pairwise import cosine_similarity

print("=" * 40)
print("SIMILARITY ANALYSIS")
print("=" * 40)

sentences = [
    "SAP has a policy on artificial intelligence ethics",  
    "The company created guidelines for AI fairness",      
    "Pizza is my favorite food",                           
    "Remote work requires secure internet connection"      
]

# calculate the similarity matrix 
similarity_matrix = cosine_similarity(embeddings)

print("\n Sentences:")
for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence[:50]}")

print("\n Similarity Matrix (1.0 = identical, 0.0 = unrelated, -1.0= opposite :")
print("\n", end="")
for i in range(len(sentences)):
    print(f" [{i}]", end = "")
print()

for i in range(len(sentences)):
    print(f"[{i}]", end = "")
    for j in range (len(sentences)):
        print(f"{similarity_matrix[i][j]:.3f} " ,end ="")
    print()

print("\n Key Observations:")
print(f"Sentence 0 vs 1 (both about AI ethics): {similarity_matrix[1][0]:.3f}")
print(f"Sentence 0 vs 2 (AI vs Pizza):          {similarity_matrix[1][2]:.3f}")
print(f"Sentence 0 vs 3 (AI vs Remote work):    {similarity_matrix[1][3]:.3f}")




SIMILARITY ANALYSIS

 Sentences:
0: SAP has a policy on artificial intelligence ethics
1: The company created guidelines for AI fairness
2: Pizza is my favorite food
3: Remote work requires secure internet connection

 Similarity Matrix (1.0 = identical, 0.0 = unrelated, -1.0= opposite :

 [0] [1] [2] [3]
[0]1.000 0.657 -0.092 0.086 
[1]0.657 1.000 -0.042 0.002 
[2]-0.092 -0.042 1.000 0.024 
[3]0.086 0.002 0.024 1.000 

 Key Observations:
Sentence 0 vs 1 (both about AI ethics): 0.657
Sentence 0 vs 2 (AI vs Pizza):          -0.042
Sentence 0 vs 3 (AI vs Remote work):    0.002


### Let's Chunk!

In [4]:
## Chunking with tiktoken

import tiktoken

print(" Document Chunking!")
print("=" * 50)

# Initialize the tokenizer
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text( text, chunk_size = 512, overlap= 128):
    """
    Splits text into overlapping chunks based on tokens.

    Args: text: String to chunk; chunk_size: Max tokens per chunk; 
    overlap: No.of tokens to overlap between chunks

    Returns: List of text chunks
    """
    # Tokenize the entire text
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        # Get the chunk of tokens
        end = start + chunk_size
        chunk_tokens = tokens[start:end]

        chunk_str = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_str)

        # Move start position
        start = start + chunk_size - overlap

        # break if we've covered all tokens
        if end >= len(tokens):
            break
            
    return chunks




 Document Chunking!


In [5]:
# Test with sample text
test_text = """
SAP's Global AI Ethics Policy establishes principles for responsible artificial intelligence development and deployment. The policy emphasizes fairness, transparency, and accountability in all AI systems. Teams must conduct bias assessments before deploying AI models. Data privacy must be maintained throughout the AI lifecycle. Human oversight is required for high-risk AI decisions. Regular audits ensure compliance with ethical standards. Training programs help employees understand AI ethics. The policy applies to all SAP products and services utilizing AI technology.
"""
print(" Original Text: ")
print(f"Characters: {len(test_text)}")
print(f"Tokens: {len(tokenizer.encode(test_text))}")

print("\n Chunking with size=100, overlap=20..")
chunks = chunk_text(test_text, chunk_size=100, overlap=20)

print(f" Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    tokens_in_chunk = len(tokenizer.encode(chunk))
    print(f"\n Chunk{i+1} ({tokens_in_chunk} tokens):")
    print(f"{chunk[:150]}..")
    

 Original Text: 
Characters: 576
Tokens: 91

 Chunking with size=100, overlap=20..
 Created 1 chunks

 Chunk1 (91 tokens):

SAP's Global AI Ethics Policy establishes principles for responsible artificial intelligence development and deployment. The policy emphasizes fairne..


In [6]:
## Load all documents and chunk them

from pathlib import Path
from PyPDF2 import PdfReader

print(" LOADING & CHUNKING ALL DOCS")
print("=" * 50)

def load_document(file_path):
    """ Load a document and extract its text.

    Args: file_path: Path object pointing to the file

    Returns: dict with 'text' and 'metadata' keys

    """

    # Convert to Path object 
    file_path = Path(file_path)

    # Get file extension
    ext = file_path.suffix.lower()

    # Initialize the variables
    text = ""
    doc_type = ""

    ## conditions
    if ext == '.pdf':
        doc_type = "pdf"
        reader = PdfReader(file_path)
        for page in reader.pages:
            text += page.extract_text()

    elif ext == '.txt':
        doc_type = "txt"
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()

    else:
        raise ValueError(f"Unsupported file type: {ext}")

     # Return text and metadata
    return {
        'text': text,
        'metadata': {
            'source': file_path.name,
            'type': doc_type,
            'path': str(file_path),
            'chars': len(text)
        }
    }

# Load all docs
folder = Path("../data/documents")
all_files = list(folder.rglob("*.pdf")) + list(folder.rglob("*.txt"))
print(f" Found {len(all_files)} documents\n")

# Load and chunk each document
all_chunks = []
chunk_metadata = []

for file_path in all_files:
    print(f"Processing: {file_path.name}...")
    # Load document
    doc = load_document(file_path)
    # Chunk the text
    chunks = chunk_text(doc['text'], chunk_size=512, overlap=128)

    # Store the chunks with metadata
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
        'source': doc['metadata']['source'],
        'chunk_id': i,
        'total_chunks': len(chunks),
        'doc_type': doc['metadata']['type']
            })

    print(f" Created {len(chunks)} chunks")

# Summary
print(f"  Total documents: {len(all_files)}")
print(f"  Total chunks: {len(all_chunks)}")
print(f"  Average chunks per doc: {len(all_chunks) / len(all_files):.1f}")

# Sample chunks
print(f"\n📄 Sample chunks:")
for i in range(min(3, len(all_chunks))):
    print(f"\nChunk {i+1} from {chunk_metadata[i]['source']}:")
    print(f"{all_chunks[i][:200]}...")

 LOADING & CHUNKING ALL DOCS
 Found 8 documents

Processing: Global_CoE_BizConduct.pdf...
 Created 37 chunks
Processing: SAP_Partner_CoC.pdf...
 Created 16 chunks
Processing: Global_AI_Ethics_Policy.pdf...
 Created 16 chunks
Processing: remote_work_policy.txt...
 Created 1 chunks
Processing: git_workflow.txt...
 Created 1 chunks
Processing: deployment_checklist.txt...
 Created 1 chunks
Processing: api_development_standards.txt...
 Created 1 chunks
Processing: security_guidelines.txt...
 Created 1 chunks
  Total documents: 8
  Total chunks: 74
  Average chunks per doc: 9.2

📄 Sample chunks:

Chunk 1 from Global_CoE_BizConduct.pdf:
Global Code of Ethics and Business 
Conduct for Employees
Version 2.1 (March 2025) | PUBLIC1 Intr oduction: A Message from Our CEO
2 Ethical Bus iness and You
2.1 Bring Out Your Best  ...................

Chunk 2 from Global_CoE_BizConduct.pdf:
8 Export Contr ols and Trade Sanctions  .........  29
5.9 Anti-Money L aundering and Commitment  
to Combating the Fi

In [7]:
## Generate Embeddings

print("Generating Embeddings for all chunks")
print("=" * 50)

print(f" Processing {len(all_chunks)} chunks...")

# Generate embeddings for all chunks 
import time
start_time = time.time()

chunk_embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=32 # Process 32 chunks at a time
)

end_time = time.time()
elapsed = end_time - start_time

print(f"\n Generated {len(chunk_embeddings)} embeddings")
print(f"Time taken: {elapsed:.2f} seconds")
print(f" Speed: {len(chunk_embeddings) / elapsed:.1f} chunks/sec")

# Show embedding details
print(f"\n Embedding details:")
print(f" Shape: {chunk_embeddings.shape}")
print(f"  Dimensions per chunk: {chunk_embeddings.shape[1]}")
print(f"  Total numbers: {chunk_embeddings.shape[0] * chunk_embeddings.shape[1]:,}")
print(f"  Memory size: {chunk_embeddings.nbytes / 1024:.1f} KB")

# Sample embeddings
print(f"\n Sample Embedding (first 10 value of chunk-1):")
print(chunk_embeddings[0][:10])

Generating Embeddings for all chunks
 Processing 74 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


 Generated 74 embeddings
Time taken: 5.67 seconds
 Speed: 13.0 chunks/sec

 Embedding details:
 Shape: (74, 384)
  Dimensions per chunk: 384
  Total numbers: 28,416
  Memory size: 111.0 KB

 Sample Embedding (first 10 value of chunk-1):
[-0.02340695  0.04694174 -0.02906264 -0.12957017 -0.00739111  0.06352971
  0.09265119  0.03446115 -0.01438221  0.01848024]


In [8]:
## Set up ChromaDB vector DB

import chromadb
from chromadb.config import Settings

## Initialize the ChromaDB client (persistent storage)
client = chromadb.PersistentClient(
    path = "../data/vector_db" # Save to disk
)

print("ChromaDB client initialized")
# Create or get a collection
collection_name = "sap_knowledge"
# Delete collection if it exists (for a clean start!)
try:
    client.delete_collection(collection_name)
    print(f"Deleted existing '{collection_name}' collection")
except:
    pass

# Create new collection
collection = client.create_collection(
    name = collection_name,
    metadata= {
        "hnsw:space": "cosine",
        "description": "SAP employee knowledge base - ethics policies and enterprise docs",
        "embedding_model": "all-MiniLM-L6-v2",
        "chunk_size": 512,
        "overlap": 128
    }
)

print(f" Collection collection: '{collection_name}")
print(f" Collection metadata: '{collection.metadata}")

ChromaDB client initialized
Deleted existing 'sap_knowledge' collection
 Collection collection: 'sap_knowledge
 Collection metadata: '{'description': 'SAP employee knowledge base - ethics policies and enterprise docs', 'chunk_size': 512, 'embedding_model': 'all-MiniLM-L6-v2', 'overlap': 128, 'hnsw:space': 'cosine'}


In [9]:
print("Adding Embeddings To ChromaDB")
print("=" * 50)

# Generate ID's for each chunk
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

# Sanity check - verify all lengths match
print(f"\n🔍 Data verification:")
print(f"  Documents:  {len(all_chunks)}")
print(f"  Embeddings: {len(chunk_embeddings)}")
print(f"  IDs:        {len(ids)}")
print(f"  Metadata:   {len(chunk_metadata)}")

# Check if all lengths match
if len(all_chunks) == len(chunk_embeddings) == len(ids) == len(chunk_metadata):
    print("All lengths match!")

    # Add to chromadb
    print(f"\n Adding {len(all_chunks)} chunks to collection...")

    collection.add(
         ids=ids,
        embeddings=chunk_embeddings.tolist(),  # Convert numpy to list for ChromaDB
        documents=all_chunks,
        metadatas=chunk_metadata
    )

    print(f"Added {collection.count()} chunks to vector database!")
    print(f"\n Sample entry:")
    print(f"  ID: {ids[0]}")
    print(f"  Document preview: {all_chunks[0][:100]}...")
    print(f"  Metadata: {chunk_metadata[0]}")
    print(f"  Embedding: {chunk_embeddings[0][:5]}... (384 dims)")

else:
    print("Error: Lengths don't match!")
    print("Cannot add to database. Check the data")
    
        


Adding Embeddings To ChromaDB

🔍 Data verification:
  Documents:  74
  Embeddings: 74
  IDs:        74
  Metadata:   74
All lengths match!

 Adding 74 chunks to collection...
Added 74 chunks to vector database!

 Sample entry:
  ID: chunk_0
  Document preview: Global Code of Ethics and Business 
Conduct for Employees
Version 2.1 (March 2025) | PUBLIC1 Intr od...
  Metadata: {'source': 'Global_CoE_BizConduct.pdf', 'chunk_id': 0, 'total_chunks': 37, 'doc_type': 'pdf'}
  Embedding: [-0.02340695  0.04694174 -0.02906264 -0.12957017 -0.00739111]... (384 dims)


In [12]:
count = collection.count()
print(f"Total items in collection: {count}")

# First 3 items in the DB
peek_data = collection.peek(limit=3)
for i in range(3):
    print(f"ID: {peek_data['ids'][i]}")
    print(f"Metadata: {peek_data['metadatas'][i]}")
    print(f"Text: {peek_data['documents'][i][:100]}...") # Show first 100 chars
    print(f"Source: {peek_data['metadatas'][i]['source']}")
    print("-" * 20)

# Run a semantic query to check the vectors working
query = "What are the rules about AI Bias prevention?"
print(f"\n ---Testing Query: {query}' ---")

print("Embedding Query!....")
query_embedding = model.encode([query]) 

# Query using embeddings 
results = collection.query(
    query_embeddings= query_embedding.tolist(),
    n_results = 3 # Return top 3 matches
    )

print(f" Found {len(results['ids'][0])} matches\n")

for i in range(len(results['ids'][0])):
    distance = results['distances'][0][i]
    similarity = 1 - distance # gives cosine similarity
    print(f"\nMatch {i+1}:")
    print(f"Source: {results['metadatas'][0][i]['source']}")
    print(f"Chunk: {results['metadatas'][0][i]['chunk_id'] + 1} of {results['metadatas'][0][i]['total_chunks']}")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print(f"Similarity Score: {similarity:.4f}")
    print(f"\n Content:")
    print(results['documents'][0][i][:300] + "...")
    print()

print("Semantic search working! Vector DB is operational!")

Total items in collection: 74
ID: chunk_0
Metadata: {'doc_type': 'pdf', 'chunk_id': 0, 'total_chunks': 37, 'source': 'Global_CoE_BizConduct.pdf'}
Text: Global Code of Ethics and Business 
Conduct for Employees
Version 2.1 (March 2025) | PUBLIC1 Intr od...
Source: Global_CoE_BizConduct.pdf
--------------------
ID: chunk_1
Metadata: {'total_chunks': 37, 'chunk_id': 1, 'source': 'Global_CoE_BizConduct.pdf', 'doc_type': 'pdf'}
Text: 8 Export Contr ols and Trade Sanctions  .........  29
5.9 Anti-Money L aundering and Commitment  
to...
Source: Global_CoE_BizConduct.pdf
--------------------
ID: chunk_2
Metadata: {'source': 'Global_CoE_BizConduct.pdf', 'doc_type': 'pdf', 'total_chunks': 37, 'chunk_id': 2}
Text:   
constant, one thing remains certain: ethics,  
integrity, and compliance drive the trust that  
i...
Source: Global_CoE_BizConduct.pdf
--------------------

 ---Testing Query: What are the rules about AI Bias prevention?' ---
Embedding Query!....
 Found 3 matches


Match 1:
Source: 